In [1]:
import lancedb
from lancedb.pydantic import LanceModel, Vector
from pydantic import PlainSerializer, TypeAdapter
from typing import Annotated
import uuid
from enum import Enum
from datetime import datetime
import pyarrow as pa
# import geneva
# from geneva import udf

## Schemas

In [2]:
clip_embedding_dim = 3
tower_embedding_dim = 2

class Cover(LanceModel):
    cover_id: int
    book_id: int
    isbn_13: str
    cover_url: str
    cover_embedding: Vector(clip_embedding_dim) #pyright: ignore[reportInvalidTypeForm]
    tower_embedding: Vector(tower_embedding_dim) #pyright: ignore[reportInvalidTypeForm]

class User(LanceModel):
    user_id: Annotated[uuid.UUID, PlainSerializer(lambda x: x.bytes, return_type=bytes)]
    tower_embedding: Vector(tower_embedding_dim) #pyright: ignore[reportInvalidTypeForm]

class InteractionEnum(str, Enum):
    rating = 'rating'

class Interaction(LanceModel):
    user_id: Annotated[uuid.UUID, PlainSerializer(lambda x: x.bytes, return_type=bytes)]
    cover_id: int
    type: InteractionEnum
    score: int
    timestamp: datetime

class RunlogEnum(str, Enum):
    fine_tuning = 'fine_tuning'

class Runlog(LanceModel):
    type: RunlogEnum
    last_run: datetime

In [43]:
clip_dim = 3
tower_dim = 2

cover_schema = pa.schema(
    [
        pa.field("cover_id", pa.int64(), nullable=False),
        pa.field("book_id", pa.int64(), nullable=False),
        pa.field("isbn_13", pa.string(), nullable=False),
        pa.field("cover_url", pa.string(), nullable=False),
        pa.field("cover_embedding", pa.fixed_shape_tensor(pa.float32(), (clip_dim,)), nullable=False),
        pa.field("tower_embedding", pa.list_(pa.float32(), tower_dim), nullable=False),
    ]
)

user_schema = pa.schema(
    [
        pa.field("user_id", pa.uuid(), nullable=False),
        pa.field("tower_embedding", pa.list_(pa.float32(), tower_dim), nullable=False),
    ]
)

interaction_schema = pa.schema(
    [
        pa.field("user_id", pa.uuid(), nullable=False),
        pa.field("cover_id", pa.int64(), nullable=False),
        pa.field("type", pa.string(), nullable=False),
        pa.field("score", pa.int64(), nullable=False),
        pa.field("timestamp", pa.timestamp('us'), nullable=False),
    ]
)

runlog_schema = pa.schema(
    [
        pa.field("type", pa.string(), nullable=False),
        pa.field("last_run", pa.timestamp('us'), nullable=False),
    ]
)

In [44]:
interaction_schema

user_id: extension<arrow.uuid> not null
cover_id: int64 not null
type: string not null
score: int64 not null
timestamp: timestamp[us] not null

In [45]:
import pyarrow as pa

user = User(user_id=uuid.uuid4(), tower_embedding=[0, 1])
data = [user.model_dump()]
table = pa.Table.from_pylist(data)
print(table)

pyarrow.Table
user_id: binary
tower_embedding: list<item: double>
  child 0, item: double
----
user_id: [[431A7EE996E048C793EFDC87A1FBB717]]
tower_embedding: [[[0,1]]]


In [86]:
uri = "test_lancedb"
#db = geneva.connect(uri)
db = lancedb.connect(uri)

In [87]:
cover_table = db.create_table(
    "covers", schema=cover_schema, exist_ok=True
)
user_table = db.create_table(
    "users", schema=user_schema, exist_ok=True
)
interaction_table = db.create_table(
    "interactions", schema=interaction_schema, exist_ok=True
)
runlog_table = db.create_table(
    "runlog", schema=runlog_schema, exist_ok=True
)

In [88]:
id_stats = cover_table.index_stats("cover_id_idx")
if not id_stats:
    cover_table.create_index("cover_id", config=lancedb.index.BTree(), name="cover_id_idx")

id_stats = user_table.index_stats("user_id_idx")
if not id_stats:
    user_table.create_index("user_id", config=lancedb.index.BTree(), name="user_id_idx")

user_id_stats = interaction_table.index_stats("user_id_idx")
cover_id_stats = interaction_table.index_stats("cover_id_idx")
if not user_id_stats or not cover_id_stats:
    interaction_table.create_index("user_id", config=lancedb.index.BTree(), name="user_id_idx")
    interaction_table.create_index("cover_id", config=lancedb.index.BTree(), name="cover_id_idx")

type_stats = runlog_table.index_stats("type_idx")
if not type_stats:
    runlog_table.create_index("type", config=lancedb.index.BTree(), name="type_idx")

In [89]:
interaction_table.list_indices()

[IndexConfig(name="cover_id_idx", index_type="BTree", columns=["cover_id"], index_uuid="c3164f5c-15ad-4ad1-bfef-6eba5c8bfe6d", type_url="/lance.table.BTreeIndexDetails", created_at=datetime.datetime(2026, 7, 23, 21, 14, 34, 426463, tzinfo=datetime.timezone.utc), num_indexed_rows=0, num_unindexed_rows=0, size_bytes=713, num_segments=1, index_version=0, index_details={}),
 IndexConfig(name="user_id_idx", index_type="BTree", columns=["user_id"], index_uuid="b785a15b-f746-4402-907e-3dd4c1762334", type_url="/lance.table.BTreeIndexDetails", created_at=datetime.datetime(2026, 7, 23, 21, 14, 34, 425963, tzinfo=datetime.timezone.utc), num_indexed_rows=0, num_unindexed_rows=0, size_bytes=758, num_segments=1, index_version=0, index_details={})]

## Ingesting Data

In [90]:
covers_adapter = TypeAdapter(list[Cover])

cover_list = [
    Cover(cover_id=1, book_id=2, isbn_13="1234567891011", cover_url="something.cool.com/bruh.jpg", cover_embedding=[1, 3, 9], tower_embedding=[1, 2]),
    Cover(cover_id=5, book_id=2, isbn_13="1234567891014", cover_url="something.cool.com/bruh2.jpg", cover_embedding=[15, 2, 2], tower_embedding=[3, 2])
]

In [91]:
(
    cover_table.merge_insert("cover_id")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute(covers_adapter.dump_python(cover_list))
)

MergeResult(version=3, num_updated_rows=0, num_inserted_rows=2, num_deleted_rows=0, num_attempts=1, num_rows=2)

In [92]:
cover_table.head()

pyarrow.Table
cover_id: int64 not null
book_id: int64 not null
isbn_13: string not null
cover_url: string not null
cover_embedding: extension<arrow.fixed_shape_tensor[value_type=float, shape=[3]]> not null
tower_embedding: fixed_size_list<item: float>[2] not null
  child 0, item: float
----
cover_id: [[5,1]]
book_id: [[2,2]]
isbn_13: [["1234567891014","1234567891011"]]
cover_url: [["something.cool.com/bruh2.jpg","something.cool.com/bruh.jpg"]]
cover_embedding: [[[15,2,2],[1,3,9]]]
tower_embedding: [[[3,2],[1,2]]]

In [ ]:
import torch
from lancedb.permutation import Permutation, permutation_builder

class PopularCoversDataSet(torch.utils.data.Dataset):
    def __init__(self, 
        table: lancedb.Table, cover_ids: list[int], cover_id_field: str = "cover_id", 
        embedding_field: str = "cover_embedding", max_rating: float = 5.0
    ):
        self.table = table
        self.cover_ids = cover_ids
        self.cover_id_field = cover_id_field
        self.embedding_field = embedding_field
        self.max_rating = max_rating
        self.default_user_id = uuid.UUID(int=0)
        self.default_user = self.get_default_user()
        self.rating_arr = torch.tensor([self.max_rating])
        self.perm = self.get_permutation()
    
    def __len__(self):
        return len(self.perm)

    def get_default_user(self) -> torch.Tensor:
        return torch.frombuffer(self.default_user_id.bytes_le, dtype=torch.int32).to(dtype=torch.float32).unsqueeze(0)
    
    def get_permutation(self) -> Permutation:
        permutation_tbl = (
            permutation_builder(self.table)
            .filter(f"{self.cover_id_field} IN ({', '.join([f'{id}' for id in self.cover_ids])})")
            .execute()
        )
        permutation = (
            Permutation.from_tables(self.table, permutation_tbl)
            .select_columns(["cover_id", "cover_embedding"])
        )
        return permutation

    def __getitem__(self, idx: int):
        cover = self.perm.__getitem__(idx)[0]
        item_arr = torch.tensor(cover[self.embedding_field])
        rating_arr = torch.tensor([self.max_rating])
        
        return self.default_user, item_arr, rating_arr

In [123]:
dataset = PopularCoversDataSet(cover_table, cover_ids=[5, 1])

dataloader = torch.utils.data.DataLoader(dataset, batch_size=1, shuffle=False)

for batch in dataloader:
    print(batch)

Len perm: 2
[tensor([[[0., 0., 0., 0.]]]), tensor([[15.,  2.,  2.]]), tensor([[5.]])]
Len perm: 2
[tensor([[[0., 0., 0., 0.]]]), tensor([[1., 3., 9.]]), tensor([[5.]])]


In [121]:
import torch
from lancedb.util import tbl_to_tensor

dataloader = torch.utils.data.DataLoader(
    cover_table, batch_size=1, shuffle=True
)

for batch in dataloader:
    print(batch)

TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'pyarrow.lib.ChunkedArray'>

In [103]:
import torch
from lancedb.permutation import Permutation, permutation_builder

permutation = Permutation.identity(cover_table).select_columns(["cover_id", "cover_embedding"])
print(permutation.__getitem__(0))
dataloader = torch.utils.data.DataLoader(
    permutation, batch_size=1, shuffle=True
)

for batch in dataloader:
    print(batch)

[{'cover_id': 5, 'cover_embedding': [15.0, 2.0, 2.0]}]
{'cover_id': tensor([1]), 'cover_embedding': [tensor([1.], dtype=torch.float64), tensor([3.], dtype=torch.float64), tensor([9.], dtype=torch.float64)]}
{'cover_id': tensor([5]), 'cover_embedding': [tensor([15.], dtype=torch.float64), tensor([2.], dtype=torch.float64), tensor([2.], dtype=torch.float64)]}


In [ ]:
stuff = {'cover_id': 5, 'cover_embedding': [15.0, 2.0, 2.0]}


In [ ]:
test_list = [1]
[f'{id}' for id in test_list]

TypeError: sequence item 0: expected str instance, int found

In [78]:
test_list = [1]
permutation_tbl = permutation_builder(cover_table).filter(f"cover_id IN ({', '.join([f'{id}' for id in test_list])})").execute()
permutation = Permutation.from_tables(cover_table, permutation_tbl).select_columns(["cover_id", "cover_embedding"])

dataloader = torch.utils.data.DataLoader(
    permutation, batch_size=1, shuffle=True
)

for batch in dataloader:
    print(batch)

{'cover_id': tensor([1]), 'cover_embedding': [tensor([1.], dtype=torch.float64), tensor([3.], dtype=torch.float64), tensor([9.], dtype=torch.float64)]}


In [54]:
users_adapter = TypeAdapter(list[User])

user_list = [
    User(user_id=uuid.uuid4(), tower_embedding=[1, 3]),
]

In [55]:
(
    user_table.merge_insert("user_id")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute(users_adapter.dump_python(user_list))
)

MergeResult(version=3, num_updated_rows=0, num_inserted_rows=1, num_deleted_rows=0, num_attempts=1, num_rows=1)

In [56]:
user_table.head()

pyarrow.Table
user_id: extension<arrow.uuid> not null
tower_embedding: fixed_size_list<item: float>[2] not null
  child 0, item: float
----
user_id: [[9EB87F4C539140E2A1045F86C45D3532]]
tower_embedding: [[[1,3]]]

In [57]:
interactions_adapter = TypeAdapter(list[Interaction])

interaction_list = [
    Interaction(user_id=user_list[0].user_id, cover_id=cover_list[1].cover_id, type=InteractionEnum.rating, score=4, timestamp=datetime.now())
]

In [58]:
interaction_table.add(interactions_adapter.dump_python(interaction_list))

AddResult(version=4)

In [59]:
interaction_table.head()

pyarrow.Table
user_id: extension<arrow.uuid> not null
cover_id: int64 not null
type: string not null
score: int64 not null
timestamp: timestamp[us] not null
----
user_id: [[9EB87F4C539140E2A1045F86C45D3532]]
cover_id: [[5]]
type: [["rating"]]
score: [[4]]
timestamp: [[2026-07-17 18:37:30.775944]]

In [60]:
runlog_adapter = TypeAdapter(list[Runlog])

runlog = [Runlog(type=RunlogEnum.fine_tuning, last_run=datetime.now())]

In [61]:
(
    runlog_table.merge_insert("type")
    .when_matched_update_all()
    .when_not_matched_insert_all()
    .execute(runlog_adapter.dump_python(runlog))
)

MergeResult(version=3, num_updated_rows=0, num_inserted_rows=1, num_deleted_rows=0, num_attempts=1, num_rows=1)

In [62]:
runlog_table.head()

pyarrow.Table
type: string not null
last_run: timestamp[us] not null
----
type: [["fine_tuning"]]
last_run: [[2026-07-17 18:37:31.123428]]

## Querying

In [63]:
current_user_id = user_list[0].user_id.hex
current_user_id

'9eb87f4c539140e2a1045f86c45d3532'

In [64]:
embedding = (
    user_table.search()
    .where(f"user_id = X'{current_user_id}'")
    .select(["tower_embedding"])
    .limit(1)
    .to_list()
)[0]["tower_embedding"]
embedding

[1.0, 3.0]

In [65]:
covers = (
    cover_table.search(embedding, vector_column_name="tower_embedding")
    .select(["cover_id", "book_id", "isbn_13", "cover_url", "tower_embedding", "_distance"])
    .limit(10)
    .to_list()
)
covers

[{'cover_id': 1,
  'book_id': 2,
  'isbn_13': '1234567891011',
  'cover_url': 'something.cool.com/bruh.jpg',
  'tower_embedding': [1.0, 2.0],
  '_distance': 1.0},
 {'cover_id': 5,
  'book_id': 2,
  'isbn_13': '1234567891014',
  'cover_url': 'something.cool.com/bruh2.jpg',
  'tower_embedding': [3.0, 2.0],
  '_distance': 5.0}]

In [66]:
cover_table.head()

pyarrow.Table
cover_id: int64 not null
book_id: int64 not null
isbn_13: string not null
cover_url: string not null
cover_embedding: fixed_size_list<item: float>[3] not null
  child 0, item: float
tower_embedding: fixed_size_list<item: float>[2] not null
  child 0, item: float
----
cover_id: [[5,1]]
book_id: [[2,2]]
isbn_13: [["1234567891014","1234567891011"]]
cover_url: [["something.cool.com/bruh2.jpg","something.cool.com/bruh.jpg"]]
cover_embedding: [[[15,2,2],[1,3,9]]]
tower_embedding: [[[3,2],[1,2]]]

## Adding Embeddings

In [67]:
import torch

In [ ]:
cover_table.add_columns({"random_embedding": f"arrow_cast(NULL, 'FixedSizeList({tower_embedding_dim}, Float32)')"})

AddColumnsResult(version=6)

In [76]:
cover_table.head()

pyarrow.Table
cover_id: int64 not null
book_id: int64 not null
isbn_13: string not null
cover_url: string not null
cover_embedding: fixed_size_list<item: float>[3] not null
  child 0, item: float
tower_embedding: fixed_size_list<item: float>[2] not null
  child 0, item: float
random_embedding: fixed_size_list<item: float>[2]
  child 0, item: float
----
cover_id: [[5,1]]
book_id: [[2,2]]
isbn_13: [["1234567891014","1234567891011"]]
cover_url: [["something.cool.com/bruh2.jpg","something.cool.com/bruh.jpg"]]
cover_embedding: [[[15,2,2],[1,3,9]]]
tower_embedding: [[[3,2],[1,2]]]
random_embedding: [[null,null]]

In [77]:
class CoverUpdate(LanceModel):
    cover_id: int
    random_embedding: Vector(tower_embedding_dim) #pyright: ignore[reportInvalidTypeForm]

cover_updates_adapter = TypeAdapter(list[CoverUpdate])

cover_update_list = [
    CoverUpdate(cover_id=1, random_embedding=[4, 3]),
    CoverUpdate(cover_id=5, random_embedding=[6, 1]),
]

In [78]:
(
    cover_table.merge_insert("cover_id")
    .when_matched_update_all()
    .execute(cover_updates_adapter.dump_python(cover_update_list))
)

MergeResult(version=7, num_updated_rows=2, num_inserted_rows=0, num_deleted_rows=0, num_attempts=1, num_rows=2)

In [79]:
cover_table.head()

pyarrow.Table
cover_id: int64 not null
book_id: int64 not null
isbn_13: string not null
cover_url: string not null
cover_embedding: fixed_size_list<item: float>[3] not null
  child 0, item: float
tower_embedding: fixed_size_list<item: float>[2] not null
  child 0, item: float
random_embedding: fixed_size_list<item: float>[2]
  child 0, item: float
----
cover_id: [[1,5]]
book_id: [[2,2]]
isbn_13: [["1234567891011","1234567891014"]]
cover_url: [["something.cool.com/bruh.jpg","something.cool.com/bruh2.jpg"]]
cover_embedding: [[[1,3,9],[15,2,2]]]
tower_embedding: [[[1,2],[3,2]]]
random_embedding: [[[4,3],[6,1]]]